# ⚖️ Étape 4 — Gestion du Déséquilibre (SMOTE & SMOTE-NC)
**Objectif :** Rééquilibrer les classes Churn (26.5% vs 73.5%) avant la modélisation.

| Technique | Usage |
|---|---|
| **SMOTE** | Variables numériques uniquement |
| **SMOTE-NC** | Variables mixtes (numériques + catégorielles) ← notre cas |

---
**Lancer :** `Kernel → Restart & Run All`

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE, SMOTENC
import warnings, json, os
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120
os.makedirs('../reports', exist_ok=True)

BLUE = '#4C9BE8'; ORANGE = '#E8734C'; GREEN = '#4CAF7D'
print('✅ Imports OK')

## 1. Chargement des données et sélection des variables finales

In [ ]:
df = pd.read_csv('../data/processed/churn_cleaned.csv')

# Variables finales retenues après feature selection
final_vars = [
    'tenure', 'MonthlyCharges', 'TotalCharges',          # numériques
    'Contract', 'OnlineSecurity', 'TechSupport',          # catégorielles
    'OnlineBackup', 'InternetService', 'PaymentMethod',
    'PaperlessBilling', 'SeniorCitizen', 'Partner', 'Dependents'
]

num_vars  = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_vars  = [v for v in final_vars if v not in num_vars]

df_sel = df[final_vars + ['Churn']].copy()
print(f'Shape : {df_sel.shape}')
print(f'Numériques   : {num_vars}')
print(f'Catégorielles: {cat_vars}')
print(f'\nDistribution Churn :')
print(df_sel['Churn'].value_counts())

## 2. Encodage des variables catégorielles

> **Important pour SMOTE-NC :** on encode les catégorielles en entiers MAIS on garde leurs indices séparés pour indiquer à SMOTE-NC quelles colonnes sont catégorielles.

In [ ]:
df_enc = df_sel.copy()
le_dict = {}

# Encodage des catégorielles + cible
for col in cat_vars + ['Churn']:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))
    le_dict[col] = le  # on garde les encodeurs pour décoder si besoin

X = df_enc[final_vars]
y = df_enc['Churn']

# Indices des colonnes catégorielles dans X
cat_indices = [X.columns.tolist().index(c) for c in cat_vars]
print(f'Shape X : {X.shape}')
print(f'Indices catégoriels : {cat_indices}')
print(f'Colonnes : {X.columns.tolist()}')

## 3. Split Train / Test

> **Règle critique :** On applique SMOTE **uniquement sur le train set**.  
> Appliquer SMOTE avant le split = fuite de données (data leakage) → résultats artificiellement bons.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train : {X_train.shape} | Test : {X_test.shape}')
print(f'\nDistribution y_train avant SMOTE :')
print(pd.Series(y_train).value_counts())
print(f'\nDistribution y_test (jamais touché) :')
print(pd.Series(y_test).value_counts())

## 4. SMOTE — Variables numériques uniquement

SMOTE génère des exemples synthétiques de la classe minoritaire en interpolant entre voisins proches.  
Utilisé ici sur les 3 variables numériques uniquement pour illustrer la technique.

In [ ]:
X_train_num = X_train[num_vars]

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_num, y_train)

print('=== Résultat SMOTE (num. uniquement) ===')
print(f'Avant : {pd.Series(y_train).value_counts().to_dict()}')
print(f'Après : {pd.Series(y_train_smote).value_counts().to_dict()}')
print(f'\nNouveaux échantillons générés : {len(X_train_smote) - len(X_train_num)}')

In [ ]:
# Visualisation avant/après SMOTE sur tenure vs MonthlyCharges
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, X_plot, y_plot, title in [
    (axes[0], X_train_num, y_train, 'Avant SMOTE'),
    (axes[1], pd.DataFrame(X_train_smote, columns=num_vars),
     y_train_smote, 'Après SMOTE')
]:
    for val, color, label in [(0, BLUE, 'No Churn'), (1, ORANGE, 'Churn')]:
        mask = np.array(y_plot) == val
        ax.scatter(np.array(X_plot['tenure'])[mask],
                   np.array(X_plot['MonthlyCharges'])[mask],
                   alpha=0.3, color=color, s=8, label=label)
    counts = pd.Series(y_plot).value_counts()
    ax.set_title(f'{title}\nNo={counts.get(0,0):,} | Yes={counts.get(1,0):,}', fontsize=11)
    ax.set_xlabel('tenure'); ax.set_ylabel('MonthlyCharges')
    ax.legend(fontsize=9)

plt.suptitle('SMOTE — Effet sur la distribution des classes', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../reports/fig12_smote.png', bbox_inches='tight')
plt.show()
print('✅ fig12 sauvegardée')

## 5. SMOTE-NC — Variables mixtes ✅ (méthode retenue)

**SMOTE-NC** (Nominal and Continuous) gère les variables mixtes :  
- Pour les variables **numériques** → interpolation comme SMOTE classique
- Pour les variables **catégorielles** → sélection de la modalité la plus fréquente parmi les voisins

C'est la méthode adaptée à notre dataset.

In [ ]:
smote_nc = SMOTENC(categorical_features=cat_indices, random_state=42)
X_train_res, y_train_res = smote_nc.fit_resample(X_train, y_train)

print('=== Résultat SMOTE-NC (variables mixtes) ===')
print(f'Avant : {pd.Series(y_train).value_counts().to_dict()}')
print(f'Après : {pd.Series(y_train_res).value_counts().to_dict()}')
print(f'\nNouveaux échantillons générés : {len(X_train_res) - len(X_train)}')
print(f'Shape X_train_res : {X_train_res.shape}')

In [ ]:
# Comparaison avant / après SMOTE-NC
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

labels = ['No Churn (0)', 'Churn (1)']
colors_bar = [BLUE, ORANGE]

# Avant
counts_before = pd.Series(y_train).value_counts().sort_index()
axes[0].bar(labels, counts_before.values, color=colors_bar,
            edgecolor='white', linewidth=1.5, width=0.5)
for i, v in enumerate(counts_before.values):
    axes[0].text(i, v + 30, f'{v:,}\n({v/len(y_train)*100:.1f}%)',
                 ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Avant SMOTE-NC', fontsize=12)
axes[0].set_ylabel('Nombre de samples')
axes[0].set_ylim(0, 5500)

# Après
counts_after = pd.Series(y_train_res).value_counts().sort_index()
axes[1].bar(labels, counts_after.values, color=colors_bar,
            edgecolor='white', linewidth=1.5, width=0.5)
for i, v in enumerate(counts_after.values):
    axes[1].text(i, v + 30, f'{v:,}\n({v/len(y_train_res)*100:.1f}%)',
                 ha='center', fontsize=11, fontweight='bold')
axes[1].set_title('Après SMOTE-NC', fontsize=12)
axes[1].set_ylabel('Nombre de samples')
axes[1].set_ylim(0, 5500)

plt.suptitle('SMOTE-NC — Rééquilibrage des classes (train set uniquement)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../reports/fig13_smote_nc.png', bbox_inches='tight')
plt.show()
print('✅ fig13 sauvegardée')

## 6. Sauvegarde des données rééquilibrées

In [ ]:
import pickle

# Sauvegarder les datasets pour l'étape modélisation
X_train_res_df = pd.DataFrame(X_train_res, columns=final_vars)
X_test_df      = pd.DataFrame(X_test.values, columns=final_vars)

X_train_res_df['Churn'] = y_train_res
X_test_df['Churn']      = y_test.values

X_train_res_df.to_csv('../data/processed/train_resampled.csv', index=False)
X_test_df.to_csv('../data/processed/test.csv', index=False)

# Sauvegarder aussi les encodeurs
with open('../data/processed/label_encoders.pkl', 'wb') as f:
    pickle.dump(le_dict, f)

print('✅ data/processed/train_resampled.csv')
print('✅ data/processed/test.csv')
print('✅ data/processed/label_encoders.pkl')
print(f'\nTrain rééquilibré : {X_train_res_df.shape}')
print(f'Test (intact)     : {X_test_df.shape}')
print('\n→ Prochaine étape : 05_modeling.ipynb')

## 7. 📋 Synthèse SMOTE

| | Avant SMOTE-NC | Après SMOTE-NC |
|---|---|---|
| No Churn | 4 139 (73.5%) | 4 139 (50%) |
| Churn | 1 495 (26.5%) | 4 139 (50%) |
| **Total** | **5 634** | **8 278** |

**Points clés :**
- SMOTE-NC appliqué **uniquement sur le train set** (pas de data leakage)
- Le **test set reste intact** (données réelles non synthétiques)
- SMOTE-NC utilisé car notre dataset a des **variables mixtes**

**→ Prochaine étape : `05_modeling.ipynb` — Régression Logistique, Arbre de Décision, Random Forest, AdaBoost**